# Module 2: Task 2 – The "Smart Researcher"
### **Goal**
Mastering the 2026 Unified Agent API with Mistral AI.
1. **ReAct Logic:** Autonomous tool selection.
2. **HITL Governance:** Human-in-the-Loop approval node.
3. **State Management:** Using the `"messages"` key to avoid 400 errors.

In [32]:
# !pip install langchain langchain-mistralai langchain-core

import os
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
# The 2026 unified agent factory
from langchain.agents import create_agent 

# Secure API Key Setup
os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY", "your-mistral-key-here")

# Initialize Mistral (Mistral-Large is optimized for tool calling)
llm = ChatMistralAI(model="mistral-large-latest", temperature=0)

## Code - Tool Definition with HITL Approval

In [33]:
@tool
def get_weather(location: str):
    """
    Useful for finding the current weather or temperature in a specific location.
    The agent must use this for any questions regarding weather, climate, or clothing.
    """
    # --- HITL: Manual Approval Node ---
    print(f"\n[GOVERNANCE] Mistral is requesting access to live weather data for: {location}")
    approval = input("Authorize execution? (yes/no): ")
    
    if approval.lower() != 'yes':
        return "Action blocked by the user."
    
    print(f"--- [TOOL LOG]: Fetching data... ---")
    return f"The weather in {location} is 25°C and Sunny."

tools = [get_weather]

## Code - The Corrected Agent Execution

In [34]:
# In 2026, we define the prompt directly inside create_agent via system_prompt.
# Note: No AgentExecutor wrapper is needed; create_agent returns a runnable graph.
agent = create_agent(
    model=llm, 
    tools=tools, 
    system_prompt="You are an Industrial AI Assistant. Think step-by-step and use tools for real-time data."
)

# --- THE FIX: Using the "messages" key to avoid 400 errors ---
query = "What should I wear today in Peshawar?"

# Mistral requires a role-based message list
input_payload = {
    "messages": [
        {"role": "user", "content": query}
    ]
}

print("--- STARTING AGENTIC REASONING ---")
response = agent.invoke(input_payload)

# The result is a dictionary containing the conversation history
print("\n--- FINAL ANSWER ---")
# The final message in the 'messages' list is the AI's response
print(response["messages"][-1].content)

--- STARTING AGENTIC REASONING ---

[GOVERNANCE] Mistral is requesting access to live weather data for: Peshawar
--- [TOOL LOG]: Fetching data... ---

--- FINAL ANSWER ---
In Peshawar today, the weather is **25°C and sunny**. Here’s what I’d recommend wearing:

1. **Lightweight Clothing**: Opt for breathable fabrics like cotton or linen to stay cool in the sunny weather.
2. **Sun Protection**: Wear a hat, sunglasses, and sunscreen to protect yourself from the sun.
3. **Comfortable Footwear**: Sandals or breathable shoes would be ideal for the temperature.
4. **Light Layers**: If you’ll be out in the evening, consider a light shawl or jacket as temperatures might drop slightly.

Enjoy your day! ☀️
